# FABRIC Generic Cluster - Complete Usage Guide

This notebook demonstrates all features of the `fabric-generic-cluster` module:

1. **Topology Management** - Load and validate YAML topologies
2. **Slice Deployment** - Create and configure FABRIC slices
3. **Network Configuration** - L2/L3 networks with routing
4. **SSH Setup** - Passwordless SSH with self-trust
5. **SELinux Management** - Check and configure SELinux modes
6. **Ansible Integration** - Full Ansible environment setup
7. **Inventory Generation** - Both online (from slice) and offline (from topology)

## Prerequisites

- FABRIC account with valid tokens
- `fabric-generic-cluster` module installed
- Topology YAML file (e.g., `m7.yaml`)

## 1. Setup and Imports

In [ ]:
import sys
from pathlib import Path

# Add parent directory to Python path to import modules
repo_root = Path.cwd().parent
# No longer needed - using installed package

print(f"✅ Python path configured")
print(f"   Repository root: {repo_root}")

In [ ]:
# Import modules

from fabric_generic_cluster import load_topology_from_yaml_file, SiteTopology, generate_inventory_template_from_yaml
from fabric_generic_cluster import deployment as sd
from fabric_generic_cluster import network_config as snc
from fabric_generic_cluster import ssh_setup as ssh
from fabric_generic_cluster import ansible_setup as ansible
from fabric_generic_cluster import selinux_management as selinux


print("✅ Modules imported successfully")

In [ ]:
# Define YAML directory

YAML_DIR = repo_root / "model"
print(f"✅ YAML directory: {YAML_DIR}")

## 2. Configuration Variables

Update these variables for your deployment:

In [ ]:
# Configuration
TOPOLOGY_FILE = "../model/m5-v4.yaml"  # Your topology YAML file
SLICE_NAME = "demo-cluster"  # Name for your slice

# Verify topology file exists
if not Path(TOPOLOGY_FILE).exists():
    print(f"❌ Topology file '{TOPOLOGY_FILE}' not found!")
    print(f"   Create a topology file or update TOPOLOGY_FILE variable.")
else:
    print(f"✅ Topology file '{TOPOLOGY_FILE}' found")

## 3. Load and Validate Topology

In [ ]:
# Load topology from YAML file
topology = load_topology_from_yaml_file(TOPOLOGY_FILE)

# Display topology summary
print(f"📋 Topology Summary:")
print(f"   Nodes: {len(topology.site_topology_nodes.nodes)}")
print(f"   Networks: {len(topology.site_topology_networks.networks)}")

# List nodes
print(f"\n🖥️  Nodes:")
for node in topology.site_topology_nodes.iter_nodes():
    print(f"   - {node.name} ({node.hostname}) @ {node.site} - {node.capacity.os}")

# List networks
print(f"\n🌐 Networks:")
for network in topology.site_topology_networks.iter_networks():
    print(f"   - {network.name} ({network.type})")

## 4. Check FABRIC Configuration and Existing Slices

In [ ]:
# Show FABRIC configuration
print("🔍 FABRIC Configuration:")
sd.show_config()

print("\n" + "="*70)

# Check existing slices
print("\n🔍 Existing Slices:")
sd.check_slices()

## 5. Generate Inventory Template (Before Deployment)

Generate an Ansible inventory template directly from the topology YAML, before deploying anything.
This is useful for planning and pre-configuring playbooks.

In [ ]:
# Generate inventory template with placeholder IPs
output_path = f"{SLICE_NAME}-inventory_template.ini"

generate_inventory_template_from_yaml(
    yaml_path=TOPOLOGY_FILE,
    output_path=output_path,
    use_placeholder_ips=True,
    placeholder_subnet="10.10.10.0/24"
)

# Display the generated template
print("\n📄 Generated inventory template:")
print("="*70)
with open(output_path, "r") as f:
    print(f.read())
print("="*70)

### Alternative 1: Generate inventory from topology object


In [ ]:
from fabric_generic_cluster import (
    load_topology_from_yaml_file,
    generate_ansible_inventory_from_topology,
    save_inventory_to_file
)

output_path = f"{SLICE_NAME}-inventory_template_from_topology_object.ini"

# Load topology
topology = load_topology_from_yaml_file(TOPOLOGY_FILE)

# Generate inventory (offline)
inventory_content = generate_ansible_inventory_from_topology(
    topology,
    use_placeholder_ips=True,
    placeholder_subnet="10.10.10.0/24"
)

# Save to file
save_inventory_to_file(inventory_content, output_path)

# Or just print it
print(inventory_content)

### Alternative 2: Use configured IPs from topology (no placeholders)


In [ ]:
from fabric_generic_cluster import generate_ansible_inventory_from_topology

topology = load_topology_from_yaml_file(TOPOLOGY_FILE)

# Only use IPs that are configured in the YAML
# Nodes without configured IPs will get text placeholders
inventory_content = generate_ansible_inventory_from_topology(
    topology,
    use_placeholder_ips=False  # Don't generate placeholder IPs
)

print(inventory_content)

## 6. Option A: Quick Deployment (All-in-One)

Use `deploy_and_configure_slice()` to deploy and configure everything automatically.

In [ ]:
# ============================================
# DEPLOYMENT MODE SELECTION
# ============================================

MANUAL_DEPLOYMENT = True

In [ ]:
# Deploy and configure everything in one step
if not MANUAL_DEPLOYMENT:

    slice = sd.deploy_and_configure_slice(
        topology=topology,
        slice_name=SLICE_NAME,
        configure_selinux=True,  # Apply SELinux from topology
        setup_ansible=True,       # Setup Ansible environment
        use_timestamp=True        # Add timestamp to slice name
    )

    if slice:
        print(f"\n✅ Slice '{slice.get_name()}' is ready!")
    else:
        print("\n❌ Deployment failed!")

## 6. Option B: Step-by-Step Deployment

Deploy and configure manually, step by step, for more control.

**Note:** Skip this section if you used Option A above.

In [ ]:
# Step 1: Deploy slice infrastructure
if MANUAL_DEPLOYMENT:
    print("="*70)
    print("STEP 1: Deploying Slice Infrastructure")
    print("="*70)
    
    slice = sd.deploy_topology_to_fabric(
        topology=topology,
        slice_name=SLICE_NAME,
        use_timestamp=True
    )

    if not slice:
        print("❌ Slice deployment failed!")
        raise Exception("Cannot continue without a slice")

    print(f"\n✅ Slice '{slice.get_name()}' created")

In [ ]:
# Step 2: Configure L3 networks (IP assignment + routing)
if MANUAL_DEPLOYMENT:
    print("\n" + "="*70)
    print("STEP 2: Configuring L3 Networks")
    print("="*70)
    
    sd.configure_l3_networks(slice, topology)

In [ ]:
# Step 3: Enable public routing (optional, only for IPv4Ext/IPv6Ext networks)
if MANUAL_DEPLOYMENT:
    print("\n" + "="*70)
    print("STEP 3: Enabling Public Routing (Optional)")
    print("="*70)
    
    # Check if there are any external networks in topology
    has_external = any(
        net.type in ["IPv4Ext", "IPv6Ext"] 
        for net in topology.site_topology_networks.iter_networks()
    )

    if has_external:
        sd.enable_public_routing(slice, topology)
    else:
        print("ℹ️  No external networks found, skipping public routing")

In [ ]:
# Step 4: Configure persistent network interfaces
if MANUAL_DEPLOYMENT:
    print("\n" + "="*70)
    print("STEP 4: Configuring Network Interfaces")
    print("="*70)

    snc.configure_node_interfaces(slice, topology)

In [ ]:
# Step 5: Setup passwordless SSH
if MANUAL_DEPLOYMENT:
    print("\n" + "="*70)
    print("STEP 5: Setting up Passwordless SSH")
    print("="*70)
    
    ssh.setup_passwordless_ssh(slice)

In [ ]:
# Step 6: Configure SELinux from topology
if MANUAL_DEPLOYMENT:
    print("\n" + "="*70)
    print("STEP 6: Configuring SELinux")
    print("="*70)
    
    selinux.configure_selinux_from_topology(slice, topology, persistent=True)

In [ ]:
# Step 7: Setup Ansible environment
if MANUAL_DEPLOYMENT:
    print("\n" + "="*70)
    print("STEP 7: Setting up Ansible Environment")
    print("="*70)
    
    ansible.setup_ansible_environment(
        slice=slice,
        topology=topology,
        python_version="auto",  # Auto-detect Python version
        deploy_sample=True
    )

## 7. Verify Network Configuration

In [ ]:
# Verify network interfaces on all nodes
snc.verify_node_interfaces(slice, topology)

## 8. SELinux Management

In [ ]:
# Display SELinux status summary
selinux.display_selinux_summary(slice, topology)

In [ ]:
# Optional: Set SELinux to permissive on OpenStack nodes
# (Uncomment if needed)

# set_selinux_permissive_for_openstack(slice, topology)

In [ ]:
# Optional: Set specific SELinux mode on all nodes
# (Uncomment and modify as needed)

# set_selinux_mode_all_nodes(
#     slice=slice,
#     topology=topology,
#     mode=SELinuxMode.PERMISSIVE,
#     persistent=True
# )

## 9. Generate Ansible Inventory from Deployed Slice

Generate inventory using actual IPs from the deployed slice.

In [ ]:
# Generate inventory from deployed slice (with real IPs)
inventory_from_slice = ansible.generate_ansible_inventory(slice, topology)

# Save to file
ansible.save_inventory_to_file(inventory_from_slice, "inventory_deployed.ini")

# Display
print("\n📄 Ansible Inventory (from deployed slice):")
print("="*70)
print(inventory_from_slice)
print("="*70)

## 10. Compare Template vs. Deployed Inventory

In [ ]:
# Compare the template (with placeholders) vs. actual (with real IPs)
print("📊 Inventory Comparison:\n")

print("Template (before deployment):")
print("  - File: inventory_template.ini")
print("  - Contains: Placeholder IPs or IPs from topology config")
print("  - Use: Pre-deployment planning\n")

print("Deployed (after deployment):")
print("  - File: inventory_deployed.ini")
print("  - Contains: Real IPs from FABRIC")
print("  - Use: Actual Ansible operations\n")

print("💡 Tip: Use 'diff inventory_template.ini inventory_deployed.ini' to see differences")

### Compare topology vs. deployed inventory


In [ ]:
topology = load_topology_from_yaml_file(TOPOLOGY_FILE)

# Generate template from topology (before deployment)
template_inventory = ansible.generate_ansible_inventory_from_topology(topology)
ansible.save_inventory_to_file(template_inventory, "inventory_template.ini")

# After deployment...
slice = sd.get_slice(SLICE_NAME)

# Generate actual inventory from deployed slice
actual_inventory = ansible.generate_ansible_inventory(slice, topology)
save_inventory_to_file(actual_inventory, "inventory_actual.ini")

print("Compare inventory_template.ini with inventory_actual.ini")

## 11. Test Ansible Connectivity

In [ ]:
# Test Ansible connectivity from control node to all managed nodes
connectivity_results = ansible.test_ansible_connectivity(slice, topology)

# Summary
if connectivity_results:
    successful = sum(1 for v in connectivity_results.values() if v)
    total = len(connectivity_results)
    print(f"\n📊 Connectivity: {successful}/{total} nodes reachable")
else:
    print("\n⚠️  Could not test Ansible connectivity")

## 12. Verify SSH Access Between Nodes

In [ ]:
# Get first node name for testing
first_node = list(topology.site_topology_nodes.nodes.values())[0]
first_network = None

# Find a network this node is connected to
for nic_name, iface_name, iface in first_node.get_all_interfaces():
    if iface.binding:
        first_network = iface.binding
        break

if first_network:
    print(f"Testing SSH from {first_node.name} to other nodes on {first_network}...\n")
    
    ssh_results = ssh.verify_ssh_access(
        slice=slice,
        topology=topology,
        source_hostname=first_node.name,
        network_name=first_network,
        use_ipv6=False
    )
    
    if ssh_results:
        successful = sum(1 for v in ssh_results.values() if v)
        total = len(ssh_results)
        print(f"\n📊 SSH Connectivity: {successful}/{total} nodes reachable")
else:
    print("⚠️  Could not find a network for SSH testing")

## 13. Access Ansible Control Node

Get connection information for the Ansible control node.

In [ ]:
# Find the Ansible control node
control_node = None
for node in topology.site_topology_nodes.iter_nodes():
    if node.specific.is_ansible_control():
        control_node = node
        break

if control_node:
    fab_control_node = slice.get_node(control_node.name)
    management_ip = fab_control_node.get_management_ip()
    
    print(f"\n🎛️  Ansible Control Node: {control_node.name}")
    print(f"\n📝 To connect and use Ansible:")
    print(f"\n1. SSH to control node:")
    print(f"   ssh -i <your-key> ubuntu@{management_ip}")
    print(f"\n2. Activate Ansible environment:")
    print(f"   source ~/ansible/venv/bin/activate")
    print(f"\n3. Test connectivity:")
    print(f"   cd ~/ansible")
    print(f"   ansible all_nodes -m ping")
    print(f"\n4. Run sample playbook:")
    print(f"   ansible-playbook ~/ansible/playbooks/sample.yml")
else:
    print("\n⚠️  No Ansible control node found in topology")

## 14. Generate Different Inventory Formats

Examples of generating inventories in different ways.

In [ ]:
# Example 1: Inventory from topology with specific placeholder subnet
inventory_custom_subnet = ansible.generate_ansible_inventory_from_topology(
    topology=topology,
    use_placeholder_ips=True,
    placeholder_subnet="192.168.100.0/24"
)
ansible.save_inventory_to_file(inventory_custom_subnet, "inventory_custom_subnet.ini")

# Example 2: Inventory without placeholder IPs (only configured IPs)
inventory_no_placeholders = ansible.generate_ansible_inventory_from_topology(
    topology=topology,
    use_placeholder_ips=False
)
ansible.save_inventory_to_file(inventory_no_placeholders, "inventory_no_placeholders.ini")

print("\n✅ Generated multiple inventory variants:")
print("   - inventory_template.ini (default placeholders)")
print("   - inventory_custom_subnet.ini (192.168.100.0/24)")
print("   - inventory_no_placeholders.ini (configured IPs only)")
print("   - inventory_deployed.ini (real IPs from slice)")

## 15. Retrieve Existing Slice

If you need to work with an already-deployed slice:

In [ ]:
# Retrieve an existing slice by name
# (Uncomment and modify if needed)

# existing_slice_name = "your-slice-name-here"
# slice = get_slice(existing_slice_name)

# if slice:
#     print(f"✅ Retrieved slice: {slice.get_name()}")
#     
#     # You can now use all the functions with this slice
#     display_selinux_summary(slice, topology)
#     inventory = generate_ansible_inventory(slice, topology)
#     # etc.
# else:
#     print(f"❌ Could not retrieve slice '{existing_slice_name}'")

## 16. Cleanup (Optional)

**⚠️  WARNING: This will delete your slice! Uncomment only if you're sure.**

In [ ]:
# Delete the slice
# (Uncomment to delete)

# if slice:
#     slice_name_to_delete = slice.get_name()
#     print(f"⚠️  Deleting slice: {slice_name_to_delete}")
#     
#     if delete_slice(slice_name_to_delete):
#         print(f"✅ Slice '{slice_name_to_delete}' deleted")
#     else:
#         print(f"❌ Failed to delete slice '{slice_name_to_delete}'")

## Summary

This notebook demonstrated:

✅ **Pre-Deployment**
- Load and validate topology from YAML
- Generate inventory templates before deployment
- Plan with placeholder IPs

✅ **Deployment**
- Quick all-in-one deployment
- Step-by-step manual deployment
- L3 network configuration with routing
- Public routing for external networks
- Persistent network interface configuration

✅ **SSH & Security**
- Passwordless SSH with self-trust
- SELinux management from topology
- SSH connectivity verification

✅ **Ansible**
- Full Ansible environment setup (multi-OS)
- Inventory generation (online and offline)
- Management network specification
- OS-based, site-based, and role-based groups
- Connectivity testing

✅ **Advanced Features**
- Multiple inventory generation methods
- Template vs. deployed comparison
- Flexible placeholder IP strategies
- Complete infrastructure automation

## Next Steps

1. SSH to your Ansible control node
2. Test Ansible connectivity: `ansible all_nodes -m ping`
3. Create custom playbooks in `~/ansible/playbooks/`
4. Deploy your applications using Ansible

For more information, see the [fabric-generic-cluster documentation](https://github.com/mcevik0/fabric-generic-cluster).